# Notebook 01 (Participant): Train + Generate with EngiOpt CGAN-2D

You will implement the full train-and-generate path and produce artifacts consumed by Notebook 02.


**Edit-safe start:** this notebook opens from GitHub in read-only source mode. Use **File -> Save a copy in Drive** before running edits so your changes stay in your own workspace.


## Notebook map

This notebook is written as a standalone lab chapter:
- context first,
- implementation second,
- interpretation third.

If you are following asynchronously, run cells in order and use the success checks to validate each stage before moving on.


## Standalone guide

This chapter is about **method integration under benchmark constraints**.
Success means reproducible artifacts and interpretable diagnostics, not only low training loss.


In [ ]:
# Colab/local dependency bootstrap
import sys

IN_COLAB = 'google.colab' in sys.modules
FORCE_INSTALL = False  # Set True to force install outside Colab
ENGIOPT_GIT = 'git+https://github.com/IDEALLab/EngiOpt.git@codex/dcc26-workshop-notebooks#egg=engiopt'

if IN_COLAB or FORCE_INSTALL:
    print('Installing dependencies...')
    !pip install engibench[beams2d] sqlitedict torch torchvision matplotlib pandas tqdm tyro wandb
    !pip install {ENGIOPT_GIT}
    print('Dependency install complete.')
else:
    print('Skipping install (using current environment). Set FORCE_INSTALL=True to install here.')


## Part A: Setup

Lock down runtime, seeds, and artifact paths before writing model logic.


### EngiBench vs EngiOpt roles in this notebook

- EngiBench: defines data semantics, constraints, and simulator objective.
- EngiOpt: defines the generative model family and training dynamics.

Keep this separation explicit in your reasoning and reporting.


### Step 1 - Configure reproducible environment

Set all global controls once; downstream cells should rely on these values only.


In [ ]:
import json
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch as th
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from engibench.problems.beams2d.v0 import Beams2D

try:
    from engiopt.cgan_2d.cgan_2d import Generator as EngiOptCGAN2DGenerator
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        'Could not import engiopt model class. Run the bootstrap cell first; on Colab, restart runtime after install if needed.'
    ) from exc

USE_WANDB_ARTIFACTS = False
WANDB_PROJECT = 'dcc26-workshop'
WANDB_ENTITY = None
WANDB_ARTIFACT_NAME = 'dcc26_beams2d_generated_artifacts'
WANDB_ARTIFACT_ALIAS = 'latest'
WANDB_LOG_TRAINING = True


def resolve_artifact_dir(create: bool = False) -> Path:
    in_colab = 'google.colab' in sys.modules
    path = Path('/content/dcc26_artifacts') if in_colab else Path('workshops/dcc26/artifacts')
    if create:
        path.mkdir(parents=True, exist_ok=True)
    return path


SEED = 7
random.seed(SEED)
np.random.seed(SEED)
th.manual_seed(SEED)
if th.cuda.is_available():
    th.cuda.manual_seed_all(SEED)

DEVICE = th.device('cuda' if th.cuda.is_available() else 'cpu')
print('device:', DEVICE)

ARTIFACT_DIR = resolve_artifact_dir(create=True)
print('artifact dir:', ARTIFACT_DIR)

CKPT_PATH = ARTIFACT_DIR / 'engiopt_cgan2d_generator_supervised.pt'
HISTORY_PATH = ARTIFACT_DIR / 'training_history.csv'
TRAIN_CURVE_PATH = ARTIFACT_DIR / 'training_curve.png'
LATENT_DIM = 32


### Step 2 - Build training slice from EngiBench dataset

Use a compact subset for workshop runtime; treat this as a pedagogical approximation, not final benchmark protocol.


In [ ]:
problem = Beams2D(seed=SEED)
train_ds = problem.dataset['train']
test_ds = problem.dataset['test']

condition_keys = problem.conditions_keys
print('condition keys:', condition_keys)

N_TRAIN = 512
subset_idx = np.random.default_rng(SEED).choice(len(train_ds), size=N_TRAIN, replace=False)

conds_np = np.stack([np.array(train_ds[k])[subset_idx].astype(np.float32) for k in condition_keys], axis=1)
designs_np = np.array(train_ds['optimal_design'])[subset_idx].astype(np.float32)
targets_np = (designs_np * 2.0) - 1.0

print('conditions shape:', conds_np.shape)
print('designs shape:', designs_np.shape)
print('target range:', float(targets_np.min()), 'to', float(targets_np.max()))


### Step 3 - Implement model setup (TODO)

Instantiate generator, optimizer, and loss exactly once.
Checkpoint: noise and condition tensors must align with model input dimensions.


In [ ]:
# TODO 1: Instantiate model, optimizer, loss, and noise sampler.
# Required objects:
# - model (EngiOptCGAN2DGenerator)
# - optimizer (Adam)
# - criterion (MSELoss)
# - sample_noise(batch_size)

raise NotImplementedError('Complete TODO 1 model setup')

# Completion check: calling `sample_noise(4)` should return shape (4, LATENT_DIM).


### Step 4 - Implement train/load logic (TODO)

Track loss per epoch and persist checkpoint/history outputs.
Checkpoint: you can reload and run generation without retraining.


In [ ]:
TRAIN_FROM_SCRATCH = True
EPOCHS = 8
BATCH_SIZE = 64

# TODO 2: Train or load checkpoint.
# Requirements:
# - if TRAIN_FROM_SCRATCH:
#   1) create DataLoader from (conds_np, targets_np)
#   2) run epoch loop and optimize reconstruction loss
#   3) collect train_losses list
#   4) save checkpoint to CKPT_PATH
#   5) save training history CSV to HISTORY_PATH
#   6) save training curve figure to TRAIN_CURVE_PATH
# - elif CKPT_PATH exists: load it
# - else: raise FileNotFoundError

raise NotImplementedError('Complete TODO 2 training/loading')

# Completion check: after training, CKPT_PATH and HISTORY_PATH should exist.


### Step 5 - Implement generation logic (TODO)

Generate conditioned designs on held-out test conditions.
Checkpoint: generated and baseline arrays are shape-compatible.


In [ ]:
# TODO 3: Generate designs and prepare condition records.
# Requirements:
# - sample N_SAMPLES from test dataset
# - run model(sample_noise(...), condition_tensor)
# - map tanh output back to [0, 1]
# - create:
#   gen_designs, baseline_designs, test_conds, conditions_records

raise NotImplementedError('Complete TODO 3 generation')

# Completion check: `gen_designs.shape == baseline_designs.shape` and len(conditions_records)==N_SAMPLES.


### Step 6 - Implement artifact export (TODO)

Notebook 02 expects these files as a strict handoff contract.
Treat artifact naming/format as part of the benchmark interface.


In [ ]:
# TODO 4: Save Notebook 02 artifacts and (optionally) W&B artifact.
# Required files:
# - generated_designs.npy
# - baseline_designs.npy
# - conditions.json
# Recommended extras:
# - checkpoint (.pt), training_history.csv, training_curve.png

raise NotImplementedError('Complete TODO 4 artifact export')

# Completion check: printed file paths exist and can be loaded by Notebook 02.


### Step 7 - Quick visual QA

Use this for fast sanity checks only; final judgment comes from Notebook 02 simulator metrics.


In [ ]:
# Quick visual side-by-side snapshot
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for i in range(6):
    axes[0, i].imshow(gen_designs[i], cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(f'gen {i}')
    axes[0, i].axis('off')

    axes[1, i].imshow(baseline_designs[i], cmap='gray', vmin=0, vmax=1)
    axes[1, i].set_title(f'base {i}')
    axes[1, i].axis('off')

fig.tight_layout()
plt.show()


## Troubleshooting

If a section fails, do not continue downstream. Fix locally first, then rerun the section and its immediate checks.
This notebook is intentionally staged so failures are localized.


## Next

Proceed to Notebook 02 for physics-based evaluation and benchmark interpretation.


## Takeaways

Before closing, record three points:
1. What conclusion is directly supported by your metrics?
2. What remains uncertain (and why)?
3. What extra experiment would you run next to reduce that uncertainty?
